In [1]:
# Smoke test for fit-cace-SOG.py (ChargeEq long-range renamed to SOG_potential)

import os, sys
import torch

# Ensure working directory is this notebook's folder
try:
    THIS_DIR = os.path.abspath(os.path.dirname(__file__))
except NameError:
    THIS_DIR = os.getcwd()

os.chdir(THIS_DIR)

ROOT_DIR = os.path.abspath(os.path.join(THIS_DIR, ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import cace
from cace.representations import Cace
from cace.modules import PolynomialCutoff, BesselRBF
from cace.models.atomistic import NeuralNetworkPotential

torch.set_default_dtype(torch.float32)
print("cwd:", os.getcwd())
print("root:", ROOT_DIR)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


cwd: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-NaCl
root: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq
torch: 2.6.0+cu124
cuda available: False


In [ ]:
from cace.data.extxyz_charge import get_dataset_from_extxyz_with_charge
from cace.tools import torch_geometric

cutoff = 5.29
collection = get_dataset_from_extxyz_with_charge(
    train_path=os.path.join(THIS_DIR, "NaCl.xyz"),
    cutoff=cutoff,
    valid_fraction=0.1,
    seed=1,
    atomic_energies={11: -4417.07609365649, 17: -12516.880649933015},
)
print("collection.data_key:", collection.data_key)

# 这里 collection.train 已经是 AtomicData 列表了
train_dataset = collection.train
train_loader = torch_geometric.DataLoader(
    dataset=train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=True,
)

batch = next(iter(train_loader))
print("batch keys:", sorted(batch.keys))


collection.data_key: {'energy': 'energy', 'forces': 'forces', 'charge': 'charge'}


AttributeError: 'AtomicData' object has no attribute 'get_positions'

In [12]:
print("collection.data_key:", collection.data_key)

collection.data_key: {'energy': 'energy', 'forces': 'forces', 'charge': 'charge'}


In [14]:
first_item = train_loader.dataset[0]

# 1) Data 本身的 keys（注意不要加括号）
print("first_item keys:", first_item.keys)

# 2) 转成 dict 后的 keys
d = first_item.to_dict()
print("first_item dict keys:", sorted(d.keys()))

# 3) 检查 charge 是否存在
if "charge" in d:
    print("charge shape:", d["charge"].shape, "sum:", float(d["charge"].sum()))
else:
    print("no 'charge' field in first_item.to_dict()")

first_item keys: ['edge_index', 'positions', 'shifts', 'unit_shifts', 'cell', 'atomic_numbers']
first_item dict keys: ['atomic_numbers', 'cell', 'edge_index', 'positions', 'shifts', 'unit_shifts']
no 'charge' field in first_item.to_dict()


In [15]:
from ase.io import read

atoms0 = read(os.path.join(THIS_DIR, "NaCl.xyz"), index=0)
print("atoms.info keys:", atoms0.info.keys())
print("atoms.arrays keys:", atoms0.arrays.keys())

print("energy from info:", atoms0.info.get("energy", None))

# 看 per-atom charge 实际存在于哪个 array 里
for k, v in atoms0.arrays.items():
    if "charge" in k.lower() or "q" == k.lower():
        print("candidate charge array:", k, "shape:", v.shape, "first 5:", v[:5])

atoms.info keys: dict_keys(['simulation_info'])
atoms.arrays keys: dict_keys(['numbers', 'positions'])
energy from info: None


In [8]:
# Inspect per-structure atom counts and total charges from the batch

import torch
from cace.tools.scatter import scatter_sum

# number of structures in this batch
if getattr(batch, "batch", None) is None:
    num_graphs = 1
    graph_index = torch.zeros((batch["positions"].shape[0],), dtype=torch.long)
else:
    graph_index = batch["batch"]
    num_graphs = int(graph_index.max().item()) + 1 if graph_index.numel() > 0 else 1

# atom counts per structure
ones = torch.ones((graph_index.shape[0],), device=graph_index.device, dtype=torch.float32)
num_atoms_per_graph = scatter_sum(ones, graph_index, dim=0, dim_size=num_graphs).to(torch.long)
print("num_graphs:", num_graphs)
print("num_atoms_per_graph:", num_atoms_per_graph.tolist())

# atomic_numbers sanity check
if "atomic_numbers" in batch.keys:
    Z = batch["atomic_numbers"].view(-1)
    unique_Z = sorted(set(Z.detach().cpu().tolist()))
    print("unique atomic_numbers in batch:", unique_Z)

# total charges per structure (if present)
# 你的数据里 per-atom 电荷字段名为 'charge'
if "charge" in batch.keys and batch["charge"] is not None:
    q = batch["charge"].view(-1).to(dtype=torch.float32)
    q_tot = scatter_sum(q, graph_index, dim=0, dim_size=num_graphs)
    print("total_charge_per_graph:", q_tot.detach().cpu().tolist())
else:
    print("charge not found in batch (no per-atom charges available)")


num_graphs: 4
num_atoms_per_graph: [16, 17, 16, 16]
unique atomic_numbers in batch: [11, 17]
charge not found in batch (no per-atom charges available)


In [10]:
# Build model (same logic as fit-cace-SOG.py, but no training)

Fourier_node = 18

radial_basis = BesselRBF(cutoff=cutoff, n_rbf=6, trainable=True)
cutoff_fn = PolynomialCutoff(cutoff=cutoff)

cace_representation = Cace(
    zs=[11, 17],
    n_atom_basis=2,
    embed_receiver_nodes=True,
    cutoff=cutoff,
    cutoff_fn=cutoff_fn,
    radial_basis=radial_basis,
    n_radial_basis=8,
    max_l=3,
    max_nu=3,
    num_message_passing=0,
    type_message_passing=["Bchi"],
    args_message_passing={"Bchi": {"shared_channels": False, "shared_l": False}},
    device=device,
    timeit=False,
    forward_features=["atomic_numbers"],  # 加这一行
)

sr_energy = cace.modules.atomwise.Atomwise(
    n_layers=3,
    output_key="SR_energy",
    n_hidden=[32, 16],
    use_batchnorm=False,
    add_linear_nn=True,
)

chi = cace.modules.Atomwise(
    n_layers=3,
    n_hidden=[24, 12],
    n_out=1,
    per_atom_output_key="chi",
    output_key="tot_chi",
    residual=False,
    add_linear_nn=True,
    post_process=torch.square,
    bias=False,
)

charge_eq = cace.modules.ChargeEq(
    dl=1.5,
    sigma=1.0,
    elements=[11, 17],
    feature_key="chi",
    output_key="q_eq",
    ewald_key="SOG_potential",
    system_charge=0.0,
    remove_self_interaction=True,
    aggregation_mode="sum",
    use_sog_kernel=True,
    sog_num_components=Fourier_node,
)

e_add = cace.modules.FeatureAdd(feature_keys=["SR_energy", "SOG_potential"], output_key="CACE_energy")
forces = cace.modules.Forces(energy_key="CACE_energy", forces_key="CACE_forces", calc_stress=False)

model = NeuralNetworkPotential(
    input_modules=None,
    representation=cace_representation,
    output_modules=[sr_energy, chi, charge_eq, e_add, forces],
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable params:", trainable_params)


trainable params: 820


In [9]:
# Optional: verify that the raw dataset item contains per-atom charge

first_item = train_loader.dataset[0]
keys = first_item.keys() if callable(getattr(first_item, "keys", None)) else getattr(first_item, "keys", [])
print("first_item keys:", sorted(list(keys)))
if "charge" in keys:
    print("first_item['charge'] shape:", first_item["charge"].shape, "sum:", float(first_item["charge"].sum()))


first_item keys: ['atomic_numbers', 'cell', 'edge_index', 'positions', 'shifts', 'unit_shifts']


In [5]:
import cace, inspect
print("cace module file:", cace.__file__)

from cace.data import atomic_data
import inspect
print("from_atoms source head:")
print(inspect.getsource(atomic_data.AtomicData.from_atoms).splitlines()[120:135])

cace module file: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/cace/__init__.py
from_atoms source head:
['            additional_info=additional_info,', '        )']


In [4]:
# One forward + backward pass
batch = batch.to(device)
print(sorted(model.model_outputs))
out = model(batch, training=True)
loss = out["CACE_energy"].sum() + out["CACE_forces"].pow(2).mean()


for k in ["SR_energy", "SOG_potential", "CACE_energy", "CACE_forces", "q_eq"]:
    print(k, "in out:", k in out)

print("loss:", float(loss.detach().cpu()))

loss.backward()
print("backward ok")


['CACE_energy', 'CACE_forces', 'SOG_potential', 'SR_energy', 'chi', 'q_eq', 'tot_chi']
SR_energy in out: True
SOG_potential in out: True
CACE_energy in out: True
CACE_forces in out: True
q_eq in out: True
loss: 2.2242584228515625
backward ok
